In [ ]:
"""
SCRIPT 03
Menggabungkan data desa prioritas Maluku dan Maluku Utara
dengan data centroid desa dari shapefile BIG.

Input:
    1. DesaPrioritas1_MALUKU_filter_faskes_2026-07-18.xlsx
    2. DesaPrioritas1_MALUKU_UTARA_filter_faskes_2026-07-18.xlsx
    3. Output/Desa_Centroid_Master.csv

Output:
    Output/Desa.csv

Join:
    Kode Desa dari file prioritas
    dengan
    KDDESA dari Desa_Centroid_Master.csv
"""


from pathlib import Path
import pandas as pd


# ============================================================
# PATH
# ============================================================

PROJECT_ROOT = Path.cwd().parent  # run from the Script/ folder

DATA_DIR = PROJECT_ROOT / "Data Kementrian Koordinator dan FKTP"

OUTPUT_DIR = PROJECT_ROOT / "Output"

OUTPUT_DIR.mkdir(
    exist_ok=True
)


# ============================================================
# INPUT FILE
# ============================================================

MASTER = OUTPUT_DIR / (
    "Desa_Centroid_Master.csv"
)


PRIORITY_FILES = [

    DATA_DIR / (
        "DesaPrioritas1_MALUKU_"
        "filter_faskes_2026-07-18.xlsx"
    ),

    DATA_DIR / (
        "DesaPrioritas1_MALUKU_UTARA_"
        "filter_faskes_2026-07-18.xlsx"
    )

]


# ============================================================
# OUTPUT FILE
# ============================================================

OUT = OUTPUT_DIR / (
    "Desa.csv"
)


# ============================================================
# KOLOM YANG AKAN DISIMPAN
# ============================================================

COLS = [

    "KDDESA",

    "NMDESA2",

    "NMKEC",

    "NMDATI2",

    "NMPROP",

    "Longitude",

    "Latitude"

]


# ============================================================
# NORMALISASI KODE DESA
# ============================================================

def norm(s):

    """
    Mengubah kode desa menjadi format angka saja.

    Contoh:

    81.08.07.2004
    menjadi
    8108072004

    """

    return (

        s

        .astype(str)

        .str.replace(

            r"\D",

            "",

            regex=True

        )

    )


# ============================================================
# MAIN
# ============================================================

def main():

    # --------------------------------------------------------
    # 1. BACA FILE DESA PRIORITAS
    # --------------------------------------------------------

    print(
        "Membaca file desa prioritas..."
    )


    priority_data = []


    for file in PRIORITY_FILES:

        print()

        print(
            f"Membaca: {file.name}"
        )


        df = pd.read_excel(

            file

        )


        print(

            f"Jumlah baris: {len(df)}"

        )


        priority_data.append(

            df

        )


    # --------------------------------------------------------
    # 2. GABUNGKAN DUA FILE
    # --------------------------------------------------------

    pri = pd.concat(

        priority_data,

        ignore_index=True

    )


    print()

    print(

        f"Total desa prioritas: {len(pri)}"

    )


    # --------------------------------------------------------
    # 3. CEK KOLOM KODE DESA
    # --------------------------------------------------------

    if "Kode Desa" not in pri.columns:

        raise KeyError(

            "Kolom 'Kode Desa' tidak ditemukan "
            "di file desa prioritas."

        )


    # --------------------------------------------------------
    # 4. NORMALISASI KODE DESA PRIORITAS
    # --------------------------------------------------------

    pri["_k"] = norm(

        pri["Kode Desa"]

    )


    # --------------------------------------------------------
    # 5. BACA MASTER CENTROID
    # --------------------------------------------------------

    print()

    print(

        "Membaca master centroid..."

    )


    master = pd.read_csv(

        MASTER,

        dtype=str

    )


    # --------------------------------------------------------
    # 6. NORMALISASI KODE MASTER
    # --------------------------------------------------------

    master["_k"] = norm(

        master["KDDESA"]

    )


    # --------------------------------------------------------
    # 7. KONVERSI KOORDINAT
    # --------------------------------------------------------

    master["Longitude"] = pd.to_numeric(

        master["Longitude"],

        errors="coerce"

    )


    master["Latitude"] = pd.to_numeric(

        master["Latitude"],

        errors="coerce"

    )


    # --------------------------------------------------------
    # 8. JOIN DATA
    # --------------------------------------------------------

    merged = pri.merge(

        master[

            COLS

            +

            [

                "_k"

            ]

        ],

        on="_k",

        how="left"

    )


    # --------------------------------------------------------
    # 9. PISAHKAN DATA MATCH DAN TIDAK MATCH
    # --------------------------------------------------------

    ok = merged[

        merged["KDDESA"].notna()

    ]


    miss = merged[

        merged["KDDESA"].isna()

    ]


    # --------------------------------------------------------
    # 10. SIMPAN DATA YANG BERHASIL MATCH
    # --------------------------------------------------------

    ok[

        COLS

    ].to_csv(

        OUT,

        index=False,

        encoding="utf-8"

    )


    # --------------------------------------------------------
    # 11. INFORMASI HASIL
    # --------------------------------------------------------

    print()

    print("=" * 60)

    print(

        "HASIL JOIN"

    )

    print("=" * 60)


    print(

        f"Desa prioritas total : {len(pri)}"

    )


    print(

        f"Cocok -> Desa.csv    : {len(ok)}"

    )


    print(

        f"Tidak cocok          : {len(miss)}"

    )


    # --------------------------------------------------------
    # 12. TAMPILKAN DATA YANG TIDAK MATCH
    # --------------------------------------------------------

    if len(miss) > 0:

        print()

        print(

            "DESA YANG TIDAK BERHASIL DICOCOKKAN:"

        )


        # Hanya tampilkan kolom yang tersedia
        # untuk menghindari error jika nama kolom
        # berbeda antar file.


        possible_cols = [

            "Provinsi",

            "Kabupaten/Kota",

            "Kecamatan",

            "Desa/Kelurahan",

            "Kode Desa"

        ]


        display_cols = [

            col

            for col in possible_cols

            if col in miss.columns

        ]


        print(

            miss[display_cols]

            .to_string(

                index=False

            )

        )


    # --------------------------------------------------------
    # 13. INFORMASI OUTPUT
    # --------------------------------------------------------

    print()

    print(

        "Output tersimpan di:"

    )


    print(

        OUT

    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    main()